# 01｜準備 HiDF 資料集

原始 HiDF 請依照以下結構放置：

```text
dataset/
├── real/
└── fake/
```

程式會搭配 `metadata.csv` 篩選圖片、依族群盡量平衡抽樣，接著切分 train、val、test，再使用 YOLO 擷取最大人臉。

注意：當 YOLO 找不到人臉時，圖片會被跳過並寫入失敗紀錄，不會使用中央裁切代替。


In [ ]:
# =========================
# 參數設定區
# =========================
CSV_PATH = "metadata.csv"
RAW_REAL_DIR = "dataset/real"
RAW_FAKE_DIR = "dataset/fake"
OUTPUT_DIR = "dataset_vit"
YOLO_MODEL_PATH = "yolov11n-face.pt"

TARGET_REAL_COUNT = 2500
TARGET_FAKE_COUNT = 2500
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

VALID_RACES = ["White", "Black", "Asian", "Latino", "Indian"]
FACE_CONF = 0.40
FACE_CROP_SCALE = 1.30
RANDOM_SEED = 42

# True 會清除舊 OUTPUT_DIR；為避免誤刪，預設 False
OVERWRITE_OUTPUT = False
FAILED_CSV = "hidf_failed_images.csv"


In [ ]:
import csv
import math
import random
import shutil
from pathlib import Path

import cv2
import pandas as pd
from tqdm.auto import tqdm
from ultralytics import YOLO

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def set_seed(seed):
    random.seed(seed)


def validate_ratios(train_ratio, val_ratio, test_ratio):
    total = train_ratio + val_ratio + test_ratio
    if not math.isclose(total, 1.0, abs_tol=1e-8):
        raise ValueError(f"train/val/test 比例總和必須為 1，目前為 {total}")


def prepare_output_dir(path, overwrite):
    path = Path(path)
    if path.exists():
        if not overwrite:
            raise FileExistsError(
                f"{path} 已存在。確認內容可清除後，將 OVERWRITE_OUTPUT 改成 True。"
            )
        shutil.rmtree(path)
    for split in ("train", "val", "test"):
        for label in ("fake", "real"):
            (path / split / label).mkdir(parents=True, exist_ok=True)


def load_metadata(csv_path):
    df = pd.read_csv(csv_path)
    id_col = next(
        (c for c in df.columns if c.lower() == "id"
         or "person_id" in c.lower() or c.lower().endswith("id")),
        None,
    )
    race_col = next(
        (c for c in df.columns if "race" in c.lower() or "ethnicity" in c.lower()),
        None,
    )
    if id_col is None or race_col is None:
        raise ValueError(f"無法辨識 ID 或 race 欄位，目前欄位：{list(df.columns)}")
    return {
        str(row[id_col]).strip(): str(row[race_col]).strip()
        for _, row in df.iterrows()
        if pd.notna(row[id_col]) and pd.notna(row[race_col])
    }


def partition_ids_by_race(id_to_race, valid_races):
    groups = {race: [] for race in valid_races}
    for person_id, race in id_to_race.items():
        if race in groups:
            groups[race].append(person_id)

    real_ids, fake_ids = set(), set()
    for race, ids in groups.items():
        ids = sorted(ids)
        random.shuffle(ids)
        middle = len(ids) // 2
        real_ids.update(ids[:middle])
        fake_ids.update(ids[middle:])
    return real_ids, fake_ids


def scan_hidf(real_dir, fake_dir, id_to_race, real_ids, fake_ids, valid_races):
    real = {race: [] for race in valid_races}
    fake = {race: [] for race in valid_races}

    real_dir = Path(real_dir)
    fake_dir = Path(fake_dir)
    if not real_dir.is_dir():
        raise FileNotFoundError(f"找不到 HiDF Real 資料夾：{real_dir}")
    if not fake_dir.is_dir():
        raise FileNotFoundError(f"找不到 HiDF Fake 資料夾：{fake_dir}")

    for path in sorted(real_dir.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        person_id = path.stem
        race = id_to_race.get(person_id)
        if person_id in real_ids and race in real:
            real[race].append(path)

    for path in sorted(fake_dir.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        parts = path.stem.split("_")
        if len(parts) < 2:
            continue
        face_id, body_id = parts[0], parts[1]
        face_race = id_to_race.get(face_id)
        body_race = id_to_race.get(body_id)
        if (
            face_id in fake_ids and body_id in fake_ids
            and face_race == body_race and face_race in fake
        ):
            fake[face_race].append(path)
    return real, fake


def balanced_sample(buckets, target_count):
    pools = {race: list(paths) for race, paths in buckets.items()}
    for paths in pools.values():
        random.shuffle(paths)

    selected = []
    active = [race for race, paths in pools.items() if paths]
    while len(selected) < target_count and active:
        for race in list(active):
            if len(selected) >= target_count:
                break
            if pools[race]:
                selected.append(pools[race].pop())
            if not pools[race]:
                active.remove(race)
    return selected


def split_paths(paths, train_ratio, val_ratio):
    paths = list(paths)
    random.shuffle(paths)
    n = len(paths)
    train_end = int(n * train_ratio)
    val_end = train_end + int(n * val_ratio)
    return {
        "train": paths[:train_end],
        "val": paths[train_end:val_end],
        "test": paths[val_end:],
    }


def largest_face_box(detector, image, confidence):
    results = detector.predict(image, conf=confidence, verbose=False)
    if not results or results[0].boxes is None or len(results[0].boxes) == 0:
        return None
    boxes = results[0].boxes.xyxy.detach().cpu().numpy()
    return max(boxes, key=lambda b: float((b[2] - b[0]) * (b[3] - b[1])))


def crop_square(image, box, scale):
    height, width = image.shape[:2]
    x1, y1, x2, y2 = map(float, box)
    center_x, center_y = (x1 + x2) / 2, (y1 + y2) / 2
    side = min(max(x2 - x1, y2 - y1) * scale, width, height)
    left = int(round(center_x - side / 2))
    top = int(round(center_y - side / 2))
    left = min(max(left, 0), width - int(side))
    top = min(max(top, 0), height - int(side))
    side = int(side)
    return image[top:top + side, left:left + side]


def unique_output_path(folder, source_path):
    candidate = folder / source_path.name
    counter = 1
    while candidate.exists():
        candidate = folder / f"{source_path.stem}_{counter}{source_path.suffix.lower()}"
        counter += 1
    return candidate


def process_split(detector, paths, output_dir, split, label, confidence, scale):
    failures = []
    saved = 0
    destination = Path(output_dir) / split / label
    for path in tqdm(paths, desc=f"{split}/{label}"):
        image = cv2.imread(str(path))
        if image is None:
            failures.append({"path": str(path), "split": split, "label": label, "reason": "read_failed"})
            continue
        box = largest_face_box(detector, image, confidence)
        if box is None:
            failures.append({"path": str(path), "split": split, "label": label, "reason": "no_face"})
            continue
        face = crop_square(image, box, scale)
        if face.size == 0:
            failures.append({"path": str(path), "split": split, "label": label, "reason": "crop_failed"})
            continue
        output_path = unique_output_path(destination, path)
        if not cv2.imwrite(str(output_path), face):
            failures.append({"path": str(path), "split": split, "label": label, "reason": "write_failed"})
            continue
        saved += 1
    return saved, failures


In [ ]:
# =========================
# 執行資料處理
# =========================
validate_ratios(TRAIN_RATIO, VAL_RATIO, TEST_RATIO)
set_seed(RANDOM_SEED)
prepare_output_dir(OUTPUT_DIR, OVERWRITE_OUTPUT)

id_to_race = load_metadata(CSV_PATH)
real_ids, fake_ids = partition_ids_by_race(id_to_race, VALID_RACES)
real_buckets, fake_buckets = scan_hidf(
    RAW_REAL_DIR, RAW_FAKE_DIR,
    id_to_race, real_ids, fake_ids, VALID_RACES
)

print("Real 原始數量：", {k: len(v) for k, v in real_buckets.items()})
print("Fake 原始數量：", {k: len(v) for k, v in fake_buckets.items()})

selected = {
    "real": balanced_sample(real_buckets, TARGET_REAL_COUNT),
    "fake": balanced_sample(fake_buckets, TARGET_FAKE_COUNT),
}
if len(selected["real"]) < TARGET_REAL_COUNT or len(selected["fake"]) < TARGET_FAKE_COUNT:
    raise ValueError(
        f"可用圖片不足：real={len(selected['real'])}, fake={len(selected['fake'])}"
    )

detector = YOLO(YOLO_MODEL_PATH)
all_failures = []
summary = {}
for label, paths in selected.items():
    for split, split_paths_ in split_paths(paths, TRAIN_RATIO, VAL_RATIO).items():
        saved, failures = process_split(
            detector, split_paths_, OUTPUT_DIR, split, label,
            FACE_CONF, FACE_CROP_SCALE
        )
        summary[f"{split}/{label}"] = saved
        all_failures.extend(failures)

pd.DataFrame(
    all_failures,
    columns=["path", "split", "label", "reason"],
).to_csv(FAILED_CSV, index=False, encoding="utf-8-sig")

print("完成：", summary)
print(f"失敗圖片：{len(all_failures)} 張，紀錄於 {FAILED_CSV}")
